# MedBoard — Train GeoSample U-Net Segmentation Model

**Run this notebook in Google Colab to train the GeoSample U-Net.**

GeoSample replaces standard 3×3 convolutions and MaxPooling with:
- **Geometry-guided oriented sampling** — probes aligned to tumor boundaries
- **Differential tokens** — explicit gradient and curvature cues
- **Edge-preserving downsampling** — dual-path pooling that retains boundary signals
- **Consensus skip connections** — geometric alignment between encoder and decoder

What this notebook does:
1. Set up the environment (mount Drive, clone repo, copy data)
2. Load BRISC 2025 segmentation dataset
3. Build the GeoSample U-Net
4. Train with Dice+BCE loss and early stopping
5. Save best weights to Google Drive
6. Plot training curves
7. Evaluate on test set (Dice + IoU + per-sample breakdown)
8. Visualize predictions

**Expected training time:** ~15–25 minutes on Colab T4 GPU

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
import shutil, os, sys

# ── Detect environment ───────────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Running in: {"Google Colab" if IS_COLAB else "Local (VS Code)"}')

if IS_COLAB:
    from google.colab import drive

    # Mount Google Drive
    drive.mount('/content/drive')

    # Clone or update the MedBoard repo from GitHub
    if not os.path.exists('/content/MedBoard'):
        !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
    else:
        !git -C /content/MedBoard pull

    # Copy dataset from Drive to local SSD (faster I/O during training)
    if not os.path.exists('/content/data'):
        print('Copying dataset to local SSD (~30 sec)...')
        shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
        print('Done.')
    else:
        print('Dataset already on local SSD.')

    # Install dependencies
    !pip install -q -r /content/MedBoard/requirements_colab.txt

    # Set Python path so we can import from MedBoard
    sys.path.insert(0, '/content/MedBoard')

    # Colab paths
    DATA_ROOT   = '/content/data'
    WEIGHTS_DIR = '/content/drive/MyDrive/medboard/weights'
    CONFIG_PATH = '/content/MedBoard/configs/config.yaml'

else:
    # ── Local / VS Code paths ────────────────────────────────────────
    REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
    DATA_ROOT   = os.path.join(REPO_ROOT, 'data', 'raw')
    WEIGHTS_DIR = os.path.join(REPO_ROOT, 'weights')
    CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'config.yaml')

    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

# ── Shared paths ─────────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'geosample_unet_best.pth')
FINAL_PATH      = os.path.join(WEIGHTS_DIR, 'geosample_unet_brisc.pth')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'\nSetup complete!')
print(f'  Data       → {DATA_ROOT}')
print(f'  Weights    → {WEIGHTS_DIR}')
print(f'  Checkpoint → {CHECKPOINT_PATH}')

## Cell 2 — Verify GPU

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {vram_gb:.1f} GB')
    if vram_gb >= 15:
        print('T4 detected — using optimized hyperparameters (batch=16, FP16).')
    else:
        print('Non-T4 GPU — consider reducing batch size if OOM occurs.')
else:
    print('WARNING: No GPU detected! Training will be very slow on CPU.')
    print('Go to Runtime → Change runtime type → T4 GPU')

## Cell 3 — Load Dataset

In [ ]:
from modules.seg_dataset import get_seg_dataloaders

# Hyperparameters tuned for Colab T4 (16 GB VRAM)
BATCH_SIZE  = 16   # T4 can handle 16 at 256x256 with FP16 (~5 GB)
NUM_WORKERS = 2    # Colab has 2 CPU cores

train_loader, val_loader, test_loader = get_seg_dataloaders(
    train_images_dir = f'{DATA_ROOT}/segmentation_task/train/images',
    train_masks_dir  = f'{DATA_ROOT}/segmentation_task/train/masks',
    test_images_dir  = f'{DATA_ROOT}/segmentation_task/test/images',
    test_masks_dir   = f'{DATA_ROOT}/segmentation_task/test/masks',
    batch_size  = BATCH_SIZE,
    num_workers = NUM_WORKERS,
    val_split   = 0.1,
    config_path = CONFIG_PATH,
)

## Cell 4 — Build GeoSample U-Net

In [ ]:
import yaml
from modules.geosample_unet import build_geosample_unet, count_parameters

# Load config
print(f'Loading config from: {CONFIG_PATH}')
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Build GeoSample U-Net
model = build_geosample_unet(cfg['segmentation'])
n_params = count_parameters(model)
print(f'GeoSample U-Net parameters: {n_params:,}  ({n_params/1e6:.2f}M)')

# Quick shape check
dummy = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    out = model(dummy)
print(f'\nInput:  {tuple(dummy.shape)}')
print(f'Output: {tuple(out.shape)}  (expected: (1, 1, 256, 256))')
assert out.shape == (1, 1, 256, 256), 'Shape mismatch!'
print('Shape check passed.')

## Cell 5 — Train

In [ ]:
from modules.seg_trainer import train_segmentation

history = train_segmentation(
    model           = model,
    train_loader    = train_loader,
    val_loader      = val_loader,
    epochs          = 40,       # max epochs (early stopping usually triggers ~20-25)
    learning_rate   = 1e-4,     # Adam with weight_decay=1e-5
    patience        = 7,        # stop if val loss doesn't improve for 7 epochs
    checkpoint_path = CHECKPOINT_PATH,
    device_str      = 'cuda',
)

## Cell 6 — Save Final Weights to Drive

In [ ]:
import shutil

# Copy best checkpoint to final weights path
shutil.copy(CHECKPOINT_PATH, FINAL_PATH)
print(f'Final weights saved to: {FINAL_PATH}')
print(f'Best val dice: {max(history["val_dice"]):.4f}  ({max(history["val_dice"])*100:.2f}%)')

## Cell 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss curve ───────────────────────────────────────────────────────
ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', color='royalblue', linewidth=2)
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2)
ax1.set_title('Loss per Epoch', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Dice + BCE Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Dice score curve ─────────────────────────────────────────────────
ax2.plot(epochs_ran, history['train_dice'], label='Train Dice', color='royalblue', linewidth=2)
ax2.plot(epochs_ran, history['val_dice'],   label='Val Dice',   color='tomato',    linewidth=2)
ax2.axhline(0.75, color='green', linestyle='--', alpha=0.5, label='Good (0.75)')
ax2.axhline(0.85, color='gold',  linestyle='--', alpha=0.5, label='Excellent (0.85)')
ax2.set_title('Dice Score per Epoch', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_ylim([0, 1])
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.suptitle('GeoSample U-Net Training — MedBoard', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save plot
plot_path = f'{WEIGHTS_DIR}/geosample_training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {plot_path}')

## Cell 8 — Evaluate on Test Set (Dice + IoU)

In [ ]:
from modules.seg_trainer import validate_one_epoch, DiceBCELoss
from torch.cuda.amp import autocast
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

# ── Overall Test Loss & Dice ─────────────────────────────────────────
criterion = DiceBCELoss()
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)

print('=' * 65)
print('          GeoSample U-Net — Test Set Evaluation')
print('=' * 65)
print(f'  Test Loss (Dice+BCE):  {test_loss:.4f}')
print(f'  Test Dice Score:       {test_dice:.4f}  ({test_dice*100:.2f}%)')
print('-' * 65)

## Cell 9 — Per-Sample Evaluation (Dice + IoU Breakdown)

In [ ]:
# ── Per-sample evaluation ────────────────────────────────────────────
all_dice_scores = []
all_iou_scores  = []

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        with autocast():
            logits = model(images)

        # Per-sample metrics
        probs = torch.sigmoid(logits)
        pred_bin = (probs > 0.5).float()

        for i in range(images.shape[0]):
            p = pred_bin[i].view(-1)
            t = masks[i].view(-1)
            inter = (p * t).sum()
            dice = (2.0 * inter + 1.0) / (p.sum() + t.sum() + 1.0)
            union = p.sum() + t.sum() - inter
            iou  = (inter + 1.0) / (union + 1.0)
            all_dice_scores.append(dice.item())
            all_iou_scores.append(iou.item())

all_dice = np.array(all_dice_scores)
all_iou  = np.array(all_iou_scores)

# ── Print summary ────────────────────────────────────────────────────
print('=' * 65)
print('     GeoSample U-Net — Per-Sample Test Metrics')
print('=' * 65)
print(f'  Total test samples:    {len(all_dice)}')
print(f'')
print(f'  Dice Score:')
print(f'    Mean:   {all_dice.mean():.4f}  ({all_dice.mean()*100:.2f}%)')
print(f'    Median: {np.median(all_dice):.4f}  ({np.median(all_dice)*100:.2f}%)')
print(f'    Std:    {all_dice.std():.4f}')
print(f'    Min:    {all_dice.min():.4f}  ({all_dice.min()*100:.2f}%)')
print(f'    Max:    {all_dice.max():.4f}  ({all_dice.max()*100:.2f}%)')
print(f'')
print(f'  IoU Score:')
print(f'    Mean:   {all_iou.mean():.4f}  ({all_iou.mean()*100:.2f}%)')
print(f'    Median: {np.median(all_iou):.4f}  ({np.median(all_iou)*100:.2f}%)')
print(f'    Std:    {all_iou.std():.4f}')
print(f'    Min:    {all_iou.min():.4f}  ({all_iou.min()*100:.2f}%)')
print(f'    Max:    {all_iou.max():.4f}  ({all_iou.max()*100:.2f}%)')
print('-' * 65)

# ── Rating distribution ──────────────────────────────────────────────
excellent = (all_dice >= 0.85).sum()
good      = ((all_dice >= 0.75) & (all_dice < 0.85)).sum()
fair      = (all_dice < 0.75).sum()
print(f'  Rating Distribution:')
print(f'    Excellent (>=0.85): {excellent:4d}  ({excellent/len(all_dice)*100:.1f}%)')
print(f'    Good (0.75-0.85):   {good:4d}  ({good/len(all_dice)*100:.1f}%)')
print(f'    Fair/Poor (<0.75):  {fair:4d}  ({fair/len(all_dice)*100:.1f}%)')
print('=' * 65)

## Cell 10 — Visualize Predictions (Random Samples)

In [ ]:
import random
import matplotlib.pyplot as plt

# ── Pick 5 random test samples ───────────────────────────────────────
test_dataset = test_loader.dataset
indices = random.sample(range(len(test_dataset)), 5)

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
fig.suptitle('GeoSample U-Net — Test Predictions', fontsize=16, fontweight='bold')

for row, idx in enumerate(indices):
    img_tensor, mask_tensor = test_dataset[idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_input)
        pred = torch.sigmoid(logits).squeeze().cpu().numpy()

    # Convert image for display (denormalize from ImageNet stats)
    img_display = img_tensor.permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_display = (img_display * std + mean).clip(0, 1)

    mask_np = mask_tensor.squeeze().numpy()
    pred_bin = (pred > 0.5).astype(float)

    # Compute sample dice
    inter = (pred_bin * mask_np).sum()
    sample_dice = (2 * inter + 1) / (pred_bin.sum() + mask_np.sum() + 1)

    # Plot: Input | Ground Truth | Prediction
    axes[row, 0].imshow(img_display)
    axes[row, 0].set_title('Input MRI', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth', fontsize=10)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred_bin, cmap='Reds', vmin=0, vmax=1)
    axes[row, 2].set_title(f'Prediction (Dice: {sample_dice:.3f})', fontsize=10)
    axes[row, 2].axis('off')

plt.tight_layout()

# Save
vis_path = f'{WEIGHTS_DIR}/geosample_predictions.png'
plt.savefig(vis_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Visualization saved to {vis_path}')

## Cell 11 — Save Metrics JSON

In [ ]:
import json

metrics = {
    'model': 'GeoSampleUNet',
    'architecture': 'GeoSample U-Net (Geometry-Guided Sampling)',
    'parameters': int(count_parameters(model)),
    'input_shape': [3, 256, 256],
    'output_shape': [256, 256, 1],
    'test_samples': len(all_dice),
    'training': {
        'epochs_ran': len(history['train_loss']),
        'best_val_dice': float(max(history['val_dice'])),
        'final_train_dice': float(history['train_dice'][-1]),
        'batch_size': BATCH_SIZE,
        'learning_rate': 1e-4,
    },
    'metrics': {
        'test_dice_score': round(float(all_dice.mean()), 4),
        'test_dice_percentage': round(float(all_dice.mean()) * 100, 2),
        'test_iou_score': round(float(all_iou.mean()), 4),
        'test_iou_percentage': round(float(all_iou.mean()) * 100, 2),
        'test_dice_std': round(float(all_dice.std()), 4),
        'test_iou_std': round(float(all_iou.std()), 4),
    },
}

metrics_path = f'{WEIGHTS_DIR}/geosample_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Metrics saved to: {metrics_path}')
print(json.dumps(metrics, indent=2))